# Sprint 007 D1 — Gross economics of the frozen trade expression

**Question:** Does the frozen midpoint trade expression contain sufficiently broad and stable gross economic margin to justify investigating its implementation shortfall?

Scope is **midpoint fills only**, primary window `2020-01-01` → `2026-07-10`, over the accepted Sprint 006 artifacts. Gate formulas are frozen in [`docs/tmp/sprint007_d1_design.md`](../../docs/tmp/sprint007_d1_design.md) and are not redefined here.

Midpoint is an optimistic gross-expression reference, **not** expected executable return. The mid→cross bridge belongs to D2.

In [ ]:
import sys
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "backtest").is_dir() and (candidate / "setup.py").exists():
            return candidate
    raise RuntimeError("Could not locate MomentumCVG repo root (set cwd or PYTHONPATH)")


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.backtest.sprint007_artifact_validation import run_d0_validation
from src.backtest.sprint007_d1_gross_margin import (
    VERDICT_BLOCKED,
    load_mid_primary_tables,
    resolve_evidence_dir,
    run_d1_analysis,
    write_d1_manifest,
    write_d1_scorecard,
    write_d1_tables,
)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

## 1. Preconditions

D0 artifact readiness is a hard prerequisite. The validated D0 result is passed into `run_d1_analysis()`. If any D0 gate fails, D1 returns `D1_BLOCKED` and this notebook stops before interpreting economics.

In [ ]:
d0 = run_d0_validation()
print("D0 verdict:", d0.verdict)
print("D0 gates all passed:", d0.all_passed)
for gate in d0.gates:
    status = "PASS" if gate.passed else "FAIL"
    print(f"{gate.gate_id} [{status}] {gate.detail}")

result = run_d1_analysis(d0_result=d0)
print("D1 verdict:", result.verdict)
if not d0.all_passed or result.verdict == VERDICT_BLOCKED:
    raise RuntimeError(
        "D1 blocked by D0 prerequisite or other precondition: "
        f"{result.manifest.get('blocker')}. Do not interpret D1 gates."
    )

bundle = load_mid_primary_tables()
included = bundle.included

## 2. Reconciliation to accepted Sprint 006 midpoint (accepted calculation)

Recomputed mid-primary aggregates versus `decision_report.json` → `by_fill.mid.primary`. Any failure here is `D1_BLOCKED`; gate results would not be interpretable.

In [ ]:
reconciliation = pd.DataFrame(result.reconciliation.as_records())
print("reconciliation passed:", result.reconciliation.passed)
reconciliation

## 3. Portfolio gross margin (accepted calculation)

Aggregate midpoint dollar P&L is the primary unit; View A mean cycle CAR is the companion portfolio view.

In [ ]:
metrics = pd.Series(result.scorecard.metrics, name="value").to_frame()
display(metrics)

daily = included.groupby("trade_date")["pnl_total"].sum().sort_index()
cumulative = daily.cumsum()

fig, ax = plt.subplots()
ax.plot(list(cumulative.index), cumulative.to_numpy(), color="#1f77b4")
ax.axhline(0.0, color="#888888", linewidth=0.8)
ax.set_title("Cumulative included midpoint P&L — primary window")
ax.set_xlabel("trade_date")
ax.set_ylabel("cumulative pnl_total ($)")
fig.autofmt_xdate()
plt.show()

## 4. Long versus short location (D1 gate statistic)

`G-Location` requires at least one side with positive P&L **and** at least 10% of included trades.

In [ ]:
sides = pd.DataFrame(result.scorecard.side_attribution)
display(sides)

fig, ax = plt.subplots(figsize=(7, 3))
colors = ["#2ca02c" if v > 0 else "#d62728" for v in sides["pnl_total"]]
ax.barh(sides["side"], sides["pnl_total"], color=colors)
ax.axvline(0.0, color="#888888", linewidth=0.8)
ax.set_title("Aggregate midpoint P&L by side")
ax.set_xlabel("pnl_total ($)")
plt.show()

## 5. Breadth (D1 gate statistic)

`G-Breadth` requires total P&L to stay positive after separately excluding the five highest-P&L dates and the five highest-P&L tickers. The ticker distribution below is exploratory context, not a gate.

In [ ]:
breadth = pd.DataFrame(result.scorecard.breadth_exclusions)
display(breadth)

ticker_pnl = included.groupby("ticker")["pnl_total"].sum().sort_values()
fig, ax = plt.subplots()
ax.hist(ticker_pnl.to_numpy(), bins=60, color="#4c72b0")
ax.axvline(0.0, color="#888888", linewidth=0.8)
ax.set_title(f"Midpoint P&L per ticker (n={len(ticker_pnl)}) — exploratory description")
ax.set_xlabel("pnl_total ($)")
ax.set_ylabel("tickers")
plt.show()

## 6. Stability across years (D1 gate statistic)

`G-Stability` requires at least two positive calendar-year buckets **and** positive total P&L after excluding the single best year.

In [ ]:
years = pd.DataFrame(result.scorecard.yearly_pnl)
display(years)

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#2ca02c" if v > 0 else "#d62728" for v in years["year_pnl"]]
ax.bar(years["year"].astype(str), years["year_pnl"], color=colors)
ax.axhline(0.0, color="#888888", linewidth=0.8)
ax.set_title("Midpoint P&L by calendar year")
ax.set_xlabel("year")
ax.set_ylabel("pnl_total ($)")
plt.show()

## 7. Scorecard and verdict

All four frozen parts must pass for `D1_CONTINUE_TO_D2`.

In [ ]:
scorecard = pd.DataFrame(
    [
        {"part_id": p.part_id, "passed": p.passed, "detail": p.detail}
        for p in result.scorecard.parts
    ]
)
display(scorecard)
print("reconciliation passed:", result.reconciliation.passed)
print("D1 VERDICT:", result.verdict)

EVIDENCE_DIR = resolve_evidence_dir()
write_d1_manifest(result, EVIDENCE_DIR / "d1_gross_margin_manifest.json")
write_d1_scorecard(result, EVIDENCE_DIR / "d1_scorecard.json")
write_d1_tables(result, EVIDENCE_DIR)
print("evidence dir:", EVIDENCE_DIR)

## 8. Limits of this evidence

- Midpoint economics are an **optimistic gross-expression reference**. They are not expected executable return, and nothing here implies midpoint fills are attainable.
- This measures the frozen `42:8` selection + CVG rule + current long/short expression + current holding period + midpoint sizing. It is **not** pure Momentum/CVG signal quality.
- Cross-fill economics, the mid→cross dollar bridge, and required execution quality are **out of scope** and belong to D2 and D3.
- No spread/liquidity filter, subgroup winner, or alternative structure was evaluated or selected.
- A `D1_CONTINUE_TO_D2` verdict only clears the sprint gate to diagnose **why** cross economics diverge; it does not validate the expression.